# DuckDB Reporting Benchmarks

In [64]:
import duckdb
import time

# Connect to the DuckDB file
con = duckdb.connect('jaffle_shop.duckdb')
con.sql('SHOW TABLES')

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ customers     │
│ orders        │
│ raw_customers │
│ raw_orders    │
│ raw_payments  │
│ stg_customers │
│ stg_orders    │
│ stg_payments  │
└───────────────┘

## Preview Tables

In [65]:
con.sql('SELECT * FROM customers LIMIT 5')

┌─────────────┬────────────┬───────────┬─────────────┬───────────────────┬──────────────────┬─────────────────────────┐
│ customer_id │ first_name │ last_name │ first_order │ most_recent_order │ number_of_orders │ customer_lifetime_value │
│    int32    │  varchar   │  varchar  │    date     │       date        │      int64       │         double          │
├─────────────┼────────────┼───────────┼─────────────┼───────────────────┼──────────────────┼─────────────────────────┤
│           1 │ Michael    │ P.        │ 2018-01-01  │ 2018-02-10        │                2 │                    33.0 │
│           3 │ Kathleen   │ P.        │ 2018-01-02  │ 2018-03-11        │                3 │                    65.0 │
│           6 │ Sarah      │ R.        │ 2018-02-19  │ 2018-02-19        │                1 │                     8.0 │
│           7 │ Martin     │ M.        │ 2018-01-14  │ 2018-01-14        │                1 │                    26.0 │
│           8 │ Frank      │ R.        │

In [66]:
con.sql('SELECT * FROM orders LIMIT 5')

┌──────────┬─────────────┬────────────┬───┬───────────────┬──────────────────────┬──────────────────┬────────┐
│ order_id │ customer_id │ order_date │ … │ coupon_amount │ bank_transfer_amount │ gift_card_amount │ amount │
│  int32   │    int32    │    date    │   │    double     │        double        │      double      │ double │
├──────────┼─────────────┼────────────┼───┼───────────────┼──────────────────────┼──────────────────┼────────┤
│        1 │           1 │ 2018-01-01 │ … │           0.0 │                  0.0 │              0.0 │   10.0 │
│        2 │           3 │ 2018-01-02 │ … │           0.0 │                  0.0 │              0.0 │   20.0 │
│        3 │          94 │ 2018-01-04 │ … │           1.0 │                  0.0 │              0.0 │    1.0 │
│        4 │          50 │ 2018-01-05 │ … │          25.0 │                  0.0 │              0.0 │   25.0 │
│        5 │          64 │ 2018-01-05 │ … │           0.0 │                 17.0 │              0.0 │   17.0 │
├

In [67]:
con.sql('SELECT * FROM raw_payments where payment_method = \'credit_card\' ORDER BY amount ASC')

┌───────┬──────────┬────────────────┬────────┐
│  id   │ order_id │ payment_method │ amount │
│ int32 │  int32   │    varchar     │ int32  │
├───────┼──────────┼────────────────┼────────┤
│    74 │       65 │ credit_card    │      2 │
│    87 │       77 │ credit_card    │      2 │
│   123 │     1337 │ credit_card    │      2 │
│   136 │     2678 │ credit_card    │      2 │
│   138 │      235 │ credit_card    │      2 │
│   170 │      590 │ credit_card    │      2 │
│   177 │     1051 │ credit_card    │      2 │
│   186 │     4203 │ credit_card    │      2 │
│   201 │      566 │ credit_card    │      2 │
│   204 │     3956 │ credit_card    │      2 │
│    ·  │       ·  │      ·         │      · │
│    ·  │       ·  │      ·         │      · │
│    ·  │       ·  │      ·         │      · │
│  3234 │     1288 │ credit_card    │   2963 │
│  4082 │     2231 │ credit_card    │   2964 │
│  5097 │      689 │ credit_card    │   2965 │
│  5017 │     4280 │ credit_card    │   2983 │
│  1735 │    

## Benchmarking Helper

In [68]:
def run_and_time_query(query: str):
    start = time.time()
    result = con.execute(query).fetchall()
    end = time.time()
    runtime_ms = round((end - start) * 1000, 2)
    return result, runtime_ms

### 1. Top 10 customers

In [69]:
query1 = '''SELECT customer_id, SUM(amount) AS total_spent
FROM orders
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10'''

In [70]:
result1, time1 = run_and_time_query(query1)
print(time1)

2.08


### 2. Monthly revenue trend

In [71]:
query2 = '''SELECT DATE_TRUNC('month', order_date) AS month, SUM(orders.amount) AS revenue
FROM orders
JOIN raw_payments USING(order_id)
GROUP BY month
ORDER BY month'''

In [72]:
result2, time2 = run_and_time_query(query2)
# print(result2)
print(time2)

1.43


### 3. Avg order value per customer

In [73]:
query3 = '''SELECT customer_id, AVG(amount) AS avg_order_value
FROM orders
GROUP BY customer_id'''

In [74]:
result3, time3 = run_and_time_query(query3)
print(time3)

2.14


### 4. Number of orders per customer

In [75]:
query4 = '''SELECT customer_id, COUNT(order_id) AS num_orders
FROM orders
GROUP BY customer_id
ORDER BY num_orders DESC'''

In [76]:
result4, time4 = run_and_time_query(query4)
# result4
print(time4)

2.05


### 5. Revenue per order

In [77]:
query5 = '''SELECT order_id, SUM(amount) AS order_revenue
FROM raw_payments
GROUP BY order_id
ORDER BY order_revenue DESC'''

In [78]:
result5, time5 = run_and_time_query(query5)
# result5
print(time5)

6.5


### 6. Active customers per month

In [79]:
query6 = '''SELECT DATE_TRUNC('month', order_date) AS month, COUNT(DISTINCT customer_id) AS active_customers
FROM orders
GROUP BY month
ORDER BY month'''

In [81]:
result6, time6 = run_and_time_query(query6)
# result6
print(time6)

3.0


### 7. Average payment per order

In [82]:
query7 = '''SELECT order_id, AVG(amount) AS avg_payment
FROM raw_payments
GROUP BY order_id'''

In [83]:
result7, time7 = run_and_time_query(query7)
# result7
print(time7)

3.66


## Query Runtime Summary

In [84]:
runtimes = [
    ('Top 10 customers', time1),
    ('Monthly revenue trend', time2),
    ('Avg order value per customer', time3),
    ('Number of orders per customer', time4),
    ('Revenue per order', time5),
    ('Active customers per month', time6),
    ('Average payment per order', time7),
]

print('\nQuery Runtime Summary (ms):')
for name, ms in runtimes:
    print(f'{name:<35} : {ms} ms')


Query Runtime Summary (ms):
Top 10 customers                    : 2.08 ms
Monthly revenue trend               : 1.43 ms
Avg order value per customer        : 2.14 ms
Number of orders per customer       : 2.05 ms
Revenue per order                   : 6.5 ms
Active customers per month          : 3.0 ms
Average payment per order           : 3.66 ms


## Polars

In [86]:
import polars as pl
import duckdb
import time

# Connect to your DuckDB database
con = duckdb.connect("jaffle_shop.duckdb")

# Load DuckDB tables into Polars
orders = pl.from_arrow(con.execute("SELECT * FROM orders").arrow())
payments = pl.from_arrow(con.execute("SELECT * FROM raw_payments").arrow())

# Ensure order_date is parsed as date
if orders.schema["order_date"] == pl.String:
    orders = orders.with_columns(pl.col("order_date").str.to_date())

# Rename 'id' → 'order_id' in orders so it can join with payments
# orders = orders.rename({"id": "order_id"})



In [57]:
print(orders.schema)

Schema([('order_id', Int32), ('customer_id', Int32), ('order_date', Date), ('status', String), ('credit_card_amount', Float64), ('coupon_amount', Float64), ('bank_transfer_amount', Float64), ('gift_card_amount', Float64), ('amount', Float64)])


In [87]:
def benchmark(func):
    start = time.time()
    result = func()
    duration = round((time.time() - start) * 1000, 2)
    return result, duration


In [88]:
# 1. Top 10 customers by total spent
def q1():
    return (
        payments.group_by("order_id")
        .agg(pl.col("amount").sum().alias("order_amount"))
        .join(orders, on="order_id")
        .group_by("customer_id")
        .agg(pl.col("order_amount").sum().alias("total_spent"))
        .sort("total_spent", descending=True)
        .limit(10)
    )

# 2. Monthly revenue trend
def q2():
    joined = orders.join(payments, on="order_id")
    return (
        joined.with_columns(pl.col("order_date").dt.truncate("1mo").alias("month"))
        .group_by("month")
        .agg(pl.col("amount").sum().alias("revenue"))
        .sort("month")
    )

# 3. Avg order value per customer
def q3():
    joined = orders.join(payments, on="order_id")
    return (
        joined.group_by("customer_id")
        .agg(pl.col("amount").mean().alias("avg_order_value"))
    )

# 4. Number of orders per customer
def q4():
    return (
        orders.group_by("customer_id")
        .agg(pl.count("order_id").alias("num_orders"))
        .sort("num_orders", descending=True)
    )

# 5. Revenue per order
def q5():
    return (
        payments.group_by("order_id")
        .agg(pl.col("amount").sum().alias("order_revenue"))
        .sort("order_revenue", descending=True)
    )

# 6. Active customers per month
def q6():
    return (
        orders.with_columns(pl.col("order_date").dt.truncate("1mo").alias("month"))
        .group_by("month")
        .agg(pl.col("customer_id").n_unique().alias("active_customers"))
        .sort("month")
    )

# 7. Average payment per order
def q7():
    return (
        payments.group_by("order_id")
        .agg(pl.col("amount").mean().alias("avg_payment"))
    )


In [ ]:
# Run benchmarks
query_funcs = [q1, q2, q3, q4, q5, q6, q7]
polars_results = []

# Define queries with labels for benchmarking
runtimes = [
    ('Top 10 customers', q1),
    ('Monthly revenue trend', q2),
    ('Avg order value per customer', q3),
    ('Number of orders per customer', q4),
    ('Revenue per order', q5),
    ('Active customers per month', q6),
    ('Average payment per order', q7),
]

# Run benchmarks and store results
polars_results = []

for name, fn in runtimes:
    _, duration = benchmark(fn)
    polars_results.append((name, duration))

# Print formatted results
print('\n📊 Polars Query Runtime Summary (ms):')
print("-" * 50)
for name, ms in polars_results:
    print(f"{name:<35} : {ms:>8.2f} ms")

    
    
    



📊 Polars Query Runtime Summary (ms):
--------------------------------------------------
Top 10 customers                    :     2.93 ms
Monthly revenue trend               :     1.26 ms
Avg order value per customer        :     0.81 ms
Number of orders per customer       :     0.55 ms
Revenue per order                   :     0.87 ms
Active customers per month          :     0.71 ms
Average payment per order           :     0.43 ms


In [93]:
# Define benchmark comparison list
benchmark_comparison = [
    ("Top 10 customers", time1, polars_results[0][1]),
    ("Monthly revenue trend", time2, polars_results[1][1]),
    ("Avg order value per customer", time3, polars_results[2][1]),
    ("Number of orders per customer", time4, polars_results[3][1]),
    ("Revenue per order", time5, polars_results[4][1]),
    ("Active customers per month", time6, polars_results[5][1]),
    ("Average payment per order", time7, polars_results[6][1])
]

# Print formatted table
print(f"{'Query':<40} | {'DuckDB (ms)':>12} | {'Polars (ms)':>12} | {'Difference (ms)':>15}")
print("-" * 85)
for name, duck, polars in benchmark_comparison:
    diff = round(duck - polars, 2)
    print(f"{name:<40} | {duck:>12.2f} | {polars:>12.2f} | {diff:>15.2f}")


Query                                    |  DuckDB (ms) |  Polars (ms) | Difference (ms)
-------------------------------------------------------------------------------------
Top 10 customers                         |         2.08 |         2.93 |           -0.85
Monthly revenue trend                    |         1.43 |         1.26 |            0.17
Avg order value per customer             |         2.14 |         0.81 |            1.33
Number of orders per customer            |         2.05 |         0.55 |            1.50
Revenue per order                        |         6.50 |         0.87 |            5.63
Active customers per month               |         3.00 |         0.71 |            2.29
Average payment per order                |         3.66 |         0.43 |            3.23


In [ ]:
# revenue per order which was a group by seems much faster in polars
# most group bys are faster in polars